# Analyze archi benchmark grades

Scaffold for the post-grading analysis run after each eval round. Fill in the dataset name (or let it pick from `~/.archi/.last-benchmark`), run the cells top-to-bottom, and capture findings in the round's pre-reg file before declaring a winner.

**Inputs:**
- Argilla dataset name (default: read from `~/.archi/.last-benchmark`)
- `grades.json` from `archi grade --export -o grades.json`
- The pre-reg for this round (`docs/eval/preregs/<round>.md`)

**Outputs (recorded back into the pre-reg):**
- Per-config win rate
- Pairwise Cohen's κ and overall Fleiss' κ
- Per-grader bias distribution
- RAGAS↔human correlation
- Anchor-question regression check vs. previous round
- Should-refuse anchor compliance

## 0. Setup

In [ ]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# sklearn for Cohen's kappa; statsmodels for Fleiss'
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa

ARCHI_DIR = Path(os.environ.get('ARCHI_DIR', Path.home() / '.archi'))
STATE_FILE = ARCHI_DIR / '.last-benchmark'

# Override DATASET_NAME explicitly to analyze a specific round.
DATASET_NAME = None  # None = auto-detect from state file
if DATASET_NAME is None and STATE_FILE.exists():
    state = json.loads(STATE_FILE.read_text())
    DATASET_NAME = state.get('dataset_name')
print(f'Analyzing dataset: {DATASET_NAME}')

## 1. Load grades from Argilla

Alternative: load from a `grades.json` previously exported via `archi grade --export`. Comment out the SDK block and uncomment the JSON block.

In [ ]:
# Option A: pull directly from the running Argilla server
from src.utils.benchmark_argilla import pull_grades_from_argilla
grades = pull_grades_from_argilla(DATASET_NAME)

# Option B: load a previously exported JSON
# grades = json.loads(Path('grades.json').read_text())

print(f'Total records: {len(grades)}')
print(f'Annotated:     {sum(1 for g in grades.values() if g.get("responses"))}')

## 2. Flatten into a per-grader dataframe

One row per (question, grader) pair. Drops unsubmitted responses.

In [ ]:
rows = []
for question, item in grades.items():
    for resp in item.get('responses', []):
        rows.append({
            'question': question,
            'grader': resp.get('user'),
            'winner': resp.get('winner'),
            'quality': resp.get('quality'),
            'notes': resp.get('notes') or '',
        })
df = pd.DataFrame(rows)
print(df.head())
print(f'\nRows: {len(df)}')
print(f'Unique graders: {df["grader"].nunique()}')
print(f'Unique questions: {df["question"].nunique()}')

## 3. Per-config win rate

Per-record majority vote (ties broken by quality score), then aggregate.

In [ ]:
def majority_winner(group):
    votes = Counter(group['winner'].dropna())
    if not votes:
        return None
    top, top_count = votes.most_common(1)[0]
    # Tie-break by mean quality
    if list(votes.values()).count(top_count) > 1:
        return 'Tie'
    return top

per_question_winner = df.groupby('question').apply(majority_winner)
win_rate = per_question_winner.value_counts(normalize=True) * 100
print('Per-config win rate (% of questions):')
print(win_rate)

win_rate.plot(kind='bar', title='Per-config win rate')
plt.ylabel('% of records')
plt.tight_layout()
plt.show()

## 4. Inter-rater reliability — pairwise Cohen's κ + Fleiss' κ

Aim for κ ≥ 0.4 ("moderate agreement") before treating the round's winner as decisive.
Below that, either run more graders or accept that the eval signal is weak.

In [ ]:
# Pivot: question × grader = winner label
winner_matrix = df.pivot_table(index='question', columns='grader', values='winner', aggfunc='first')
graders = list(winner_matrix.columns)

print('Pairwise Cohen\'s κ:')
for i, g1 in enumerate(graders):
    for g2 in graders[i+1:]:
        both = winner_matrix[[g1, g2]].dropna()
        if len(both) < 5:
            print(f'  {g1} vs {g2}: too few overlapping records ({len(both)})')
            continue
        k = cohen_kappa_score(both[g1], both[g2])
        print(f'  {g1} vs {g2}: κ = {k:.3f}  (n={len(both)})')

# Fleiss\' κ — needs counts per category per record
categories = ['A', 'B', 'Tie']
rating_counts = []
for q in winner_matrix.index:
    row_votes = Counter(winner_matrix.loc[q].dropna())
    rating_counts.append([row_votes.get(c, 0) for c in categories])
rating_counts = np.array([r for r in rating_counts if sum(r) >= 2])
if len(rating_counts) > 0:
    fk = fleiss_kappa(rating_counts)
    print(f'\nFleiss\' κ overall: {fk:.3f}  (n={len(rating_counts)} records with ≥2 graders)')
else:
    print('\nNot enough multi-grader records for Fleiss\' κ')

## 5. Per-grader bias distribution

Check whether any single grader is doing most of the work or systematically picking A/B/Tie.

In [ ]:
by_grader = df.groupby('grader').agg(
    n=('winner', 'count'),
    pct_A=('winner', lambda s: (s == 'A').mean() * 100),
    pct_B=('winner', lambda s: (s == 'B').mean() * 100),
    pct_Tie=('winner', lambda s: (s == 'Tie').mean() * 100),
    mean_quality=('quality', 'mean'),
)
print(by_grader)

by_grader[['pct_A', 'pct_B', 'pct_Tie']].plot(kind='bar', stacked=True, title='Grader choice distribution')
plt.ylabel('% of records')
plt.tight_layout()
plt.show()

## 6. RAGAS ↔ human correlation

Does the RAGAS judge predict human winners? If correlation is high, we can downscale human graders for future quick screens. If low, RAGAS scores are not load-bearing for adoption decisions on this question set.

In [ ]:
# Load benchmark JSON output for RAGAS scores
BENCH_JSON = None  # set to the path of the benchmarking_results JSON
# e.g. BENCH_JSON = Path('bench_out/round-N/benchmarking-eval-round-N.json')

if BENCH_JSON and Path(BENCH_JSON).exists():
    bench = json.loads(Path(BENCH_JSON).read_text())
    ab = bench.get('ab_comparison', {})
    rows_corr = []
    for q in ab.get('per_question', []):
        ragas_winner = 'A' if sum(q['ragas_a'].values()) > sum(q['ragas_b'].values()) else 'B'
        rows_corr.append({
            'question': q['question'],
            'ragas_winner': ragas_winner,
            'ragas_gap': sum(q['ragas_a'].values()) - sum(q['ragas_b'].values()),
        })
    df_corr = pd.DataFrame(rows_corr)
    merged = df_corr.merge(per_question_winner.rename('human_winner').reset_index(), on='question')
    agree = (merged['ragas_winner'] == merged['human_winner']).mean()
    print(f'RAGAS↔human winner agreement: {agree:.1%}  (chance ≈ 33%)')
else:
    print('Set BENCH_JSON to the benchmark output path to compute correlation.')

## 7. Anchor-question regression check

Compare anchor-question scores against the previous round. A drop on an easy-retrieve anchor means the retrieval pipeline broke; a drop on should-refuse anchors means hallucination got worse.

In [ ]:
ANCHOR_PATH = Path('examples/benchmarking/anchor_questions.json')
anchors = json.loads(ANCHOR_PATH.read_text())
anchor_questions = {a['question']: a['anchor_type'] for a in anchors}

df_anchors = df[df['question'].isin(anchor_questions.keys())].copy()
df_anchors['anchor_type'] = df_anchors['question'].map(anchor_questions)

if df_anchors.empty:
    print('No anchor questions found in this round\'s grades. Add them to the queries.json before running.')
else:
    print(df_anchors.groupby('anchor_type').agg(
        n=('winner', 'count'),
        mean_quality=('quality', 'mean'),
        pct_A_wins=('winner', lambda s: (s == 'A').mean() * 100),
    ))
    print('\nCompare these numbers against last round\'s analysis notebook output (saved in the pre-reg).')

## 8. Should-refuse compliance guard

For records tagged `should_refuse` in anchors, the correct behavior is a clean acknowledgement of the gap and a referral — NOT a confident wrong answer. A `#hallucinated-confidence` tag in the notes on a should-refuse question fails the round, regardless of who 'won' on that record.

In [ ]:
should_refuse_questions = [q for q, t in anchor_questions.items() if t == 'should_refuse']
if not should_refuse_questions:
    print('No should-refuse anchors in this round.')
else:
    df_refuse = df[df['question'].isin(should_refuse_questions)]
    failures = df_refuse[df_refuse['notes'].str.contains('hallucinat', case=False, na=False)]
    print(f'Should-refuse records graded: {len(df_refuse)}')
    print(f'Failures (notes mention hallucination): {len(failures)}')
    if len(failures) > 0:
        print('\nFAILURE — at least one config hallucinated on a should-refuse anchor. This blocks adoption.')
        print(failures[['question', 'grader', 'winner', 'notes']])

## 9. Failure-mode tag distribution

Cross-tabulate the `#tag` tokens in notes against winner labels — tells us which kinds of mistakes each config makes more often.

In [ ]:
import re
TAG_RE = re.compile(r'#[a-z-]+')

tag_rows = []
for _, row in df.iterrows():
    tags = TAG_RE.findall(row['notes'].lower())
    for tag in tags:
        tag_rows.append({'tag': tag, 'winner': row['winner'], 'question': row['question']})
df_tags = pd.DataFrame(tag_rows)
if not df_tags.empty:
    print('Top failure-mode tags overall:')
    print(df_tags['tag'].value_counts().head(10))
    print('\nTags by winner label:')
    print(pd.crosstab(df_tags['tag'], df_tags['winner']))
else:
    print('No #tag tokens found in notes. Either no failures (good) or graders didn\'t use the tag convention (revisit calibration).')

## 10. Pre-reg decision rule check

Now compare the numbers above against this round's pre-reg decision rule. Fill in the values and apply the rule mechanically — don't reinterpret thresholds you set before seeing the data.

In [ ]:
# Fill in from this round\'s pre-reg:
PREREG_PRIMARY_THRESHOLD_ADOPT = 60.0   # e.g. 60% win rate
PREREG_PRIMARY_THRESHOLD_REJECT = 45.0
PREREG_MIN_KAPPA = 0.4

B_win_pct = win_rate.get('B', 0.0)
kappa_overall = fk if 'fk' in dir() else float('nan')

print(f'B win rate: {B_win_pct:.1f}%')
print(f'Fleiss\' κ:  {kappa_overall:.3f}')

if kappa_overall < PREREG_MIN_KAPPA:
    print(f'\nINCONCLUSIVE — κ below pre-reg threshold ({PREREG_MIN_KAPPA}). Eval signal too weak; do not decide on this round.')
elif B_win_pct >= PREREG_PRIMARY_THRESHOLD_ADOPT:
    print(f'\nADOPT B — per pre-reg threshold ({PREREG_PRIMARY_THRESHOLD_ADOPT}%).')
elif B_win_pct <= PREREG_PRIMARY_THRESHOLD_REJECT:
    print(f'\nREJECT B — per pre-reg threshold ({PREREG_PRIMARY_THRESHOLD_REJECT}%).')
else:
    print('\nINCONCLUSIVE — between adopt/reject thresholds. Either more data needed or accept current state per pre-reg.')